# Explainable Purchase Intention Prediction — Live App

Run the cells **top to bottom** (click a cell, press **Shift + Enter**).
The last cell waits ~30s and prints a **public link** — open it to use the app.

Pipeline: **RISE** predicts · **SHINE** explains (SHAP) · **SAGE** recommends (LLM) · SDG 12 carbon saving.


### 1. Get the code and install the libraries  *(~2 minutes)*


In [ ]:
!git clone --depth 1 https://github.com/tushaalgadiya-hue/greensight-ui.git
%cd greensight-ui
!pip install -q -r requirements.txt
print("\nDone installing.")


### 2. (Optional) Turn on the live AI recommendation
Paste a free Groq API key between the quotes for live LLM recommendations.
Leave it empty and the app still works with a clear built-in recommendation.


In [ ]:
import os
os.environ["GROQ_API_KEY"] = ""   # <-- paste your Groq key here (optional)
print("Live AI on (Groq)" if os.environ["GROQ_API_KEY"] else "No key set - using the built-in recommendation")


### 3. Train the model  *(~1 minute, run once)*


In [ ]:
!python train_and_save.py


### 4. Launch the app and get the public link
This cell waits until the link is ready, then prints it. Just run it once.


In [ ]:
import time, re
from IPython import get_ipython

# stop anything from a previous run
get_ipython().system_raw('pkill -f streamlit 2>/dev/null')
get_ipython().system_raw('pkill -f cloudflared 2>/dev/null')
time.sleep(2)

# download the tunnel tool
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared

# start the Streamlit app in the background
get_ipython().system_raw('streamlit run app.py --server.port 8501 --server.address 0.0.0.0 --server.headless true --server.enableCORS false --server.enableXsrfProtection false &')

# open a public tunnel and WAIT until the link appears
open('/content/cf.log','w').close()
get_ipython().system_raw('/content/cloudflared tunnel --url http://localhost:8501 > /content/cf.log 2>&1 &')

print('Starting the app and opening a public link (about 30 seconds)...')
url = ''
for _ in range(45):
    time.sleep(2)
    try:
        log = open('/content/cf.log').read()
        m = re.search(r'https://[-a-z0-9.]+\.trycloudflare\.com', log)
        if m:
            url = m.group(0)
            break
    except Exception:
        pass

print('\n' + '='*48)
if url:
    print('  OPEN YOUR APP HERE:')
    print('  ' + url)
else:
    print('  Still starting - wait 15 seconds and run this cell again.')
print('='*48)


---
The link stays live while this Colab tab is open and running. If it ever stops, just run cell 4 again for a fresh link.
